# Fase 1 — Reproducir el repo oficial sobre DIVA-HisDB

Este notebook **no contiene logica**: solo orquesta el paquete `backend/paper_replica/`,
que esta cubierto por tests. Si algo falla aca, se arregla en el paquete y se agrega
el test correspondiente — no se parchea el notebook.

- Diseno: `docs/replica-denardin/2026-08-10-diseno.md`
- Plan: `docs/replica-denardin/2026-08-10-plan-fase-1.md`

**Criterio de aceptacion:** F1 **ponderado sobre las 4 clases** (la metrica de
`get_scores(average="weighted")` del repo oficial) dentro de ±0.02 de CB55 0.991 /
CS18 0.984 / CS863 0.975. Si no se cumple, no se avanza a la Fase 2.

**Runtime:** Entorno de ejecucion → Cambiar tipo de entorno → **T4 GPU**.

## 1. Entorno y datos

**Antes de correr la celda:** subi `paper_replica.zip` con el panel de archivos
(icono de carpeta en la barra izquierda → boton de subir). El repo es privado, asi
que el paquete no se puede clonar desde Colab; los datos de DIVA-HisDB si son
publicos y se clonan solos.

Para regenerar el zip desde el repo:

```bash
cd backend && zip -qr ../paper_replica.zip paper_replica \
  -x "paper_replica/.venv/*" "*/__pycache__/*" "*.pyc" "paper_replica/.pytest_cache/*"
```

In [ ]:
!pip -q install segmentation-models-pytorch==0.3.4 einops==0.8.0 scikit-image==0.24.0

import os, sys, pathlib

ZIP = '/content/paper_replica.zip'
assert pathlib.Path(ZIP).exists(), (
    'Falta paper_replica.zip.\n'
    'Subilo con el panel de archivos (icono de carpeta en la barra izquierda '
    '-> boton de subir) y volve a correr esta celda.'
)

# el repo es privado, asi que el paquete viaja como zip en vez de por git clone
!mkdir -p /content/repo/backend
!unzip -qo {ZIP} -d /content/repo/backend

# los datos si son publicos; el guard evita reclonar al reejecutar la celda
![ -d /content/datos-diva ] || git clone --depth 1 https://github.com/axelden/Few-shot-DIA-WACV2023.git /content/datos-diva

sys.path.insert(0, '/content/repo/backend')
os.environ['DIVA_ROOT'] = '/content/datos-diva/fewshot_data'

assert pathlib.Path('/content/repo/backend/paper_replica/config.py').exists(), \
    'el unzip no dejo el paquete donde corresponde'
print('paquete y datos listos')

Por defecto el paquete busca los datos en `<repo>/datos-diva/fewshot_data`. En Colab
quedan en directorios hermanos, asi que la variable de entorno `DIVA_ROOT` de la celda
anterior redirige la busqueda. Tiene que quedar seteada **antes** de importar
`paper_replica.config`.

In [ ]:
from paper_replica.config import DIVA_ROOT, MANUSCRITOS, Config

print('DIVA_ROOT:', DIVA_ROOT)
assert DIVA_ROOT.exists(), f'no encuentro los datos en {DIVA_ROOT}'

for m in MANUSCRITOS:
    for split in ('training', 'validation', 'public-test'):
        n = len(list((DIVA_ROOT / m / f'img-{m}' / split).glob('*')))
        print(f'  {m:6} {split:12} {n:2d} paginas')

## 2. Donde se guarda lo que cuesta GPU

`/content/` se borra cuando Colab recicla el runtime. El 2026-08-11 se perdieron asi los
tres modelos entrenados y hubo que reentrenar. Todo lo que cueste GPU va a Drive.

Ademas de los checkpoints se guarda **la matriz de confusion de cada manuscrito**. Es un
estadistico suficiente: de ella se recalcula cualquier metrica —precision, recall, IoU, F1,
ponderada o macro— sin el modelo y sin GPU. Si hubiera estado guardada, la corrida perdida
no habria hecho falta repetirla.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

SALIDA = pathlib.Path('/content/drive/MyDrive/replica-denardin/fase1')
SALIDA.mkdir(parents=True, exist_ok=True)
print('resultados ->', SALIDA)

# si una corrida anterior dejo checkpoints, aparecen aca y se pueden reusar
previos = sorted(SALIDA.glob('*.pt'))
print('checkpoints ya disponibles:', [p.name for p in previos] or 'ninguno')

## 3. Los tests corren primero

Antes de gastar GPU. Si algo esta roto, se sabe aca y no despues de 200 epocas.

In [ ]:
!cd /content/repo/backend && python -m pytest paper_replica/tests/ -q -k "not end_to_end"

## 4. GPU y configuracion

In [ ]:
import torch

assert torch.cuda.is_available(), 'activa la GPU: Entorno de ejecucion -> Cambiar tipo de entorno -> T4'
props = torch.cuda.get_device_properties(0)
print(f'{props.name}  {props.total_memory / 1e9:.1f} GB')

cfg = Config()
print(f'parche {cfg.patch_size}  grilla {cfg.grid_cols}x{cfg.grid_rows}='
      f'{cfg.patches_per_page}  +{cfg.n_crops} crops  micro_batch {cfg.micro_batch}')
print(f'backbone {cfg.backbone}  pesos {cfg.encoder_weights}  AMP {cfg.use_amp}')

## 5. Entrenamiento

Hasta 200 epocas por manuscrito con early stopping (paciencia 20, minimo 50 epocas).
Cada modelo se guarda en Drive apenas termina, no al final del bucle.

**Corre CB55 solo la primera vez.** Es el manuscrito mas facil y tarda ~30 min: si con la
metrica oficial da ~0.99, la configuracion es correcta y recien ahi conviene invertir las
2 h de los tres. Para eso, `A_ENTRENAR = ('CB55',)`.

Si aparece OOM: bajar a `cfg = Config(micro_batch=4)`. **Nunca** por debajo de 4, y nunca
compensar congelando BatchNorm ni activando AMP — ese fue el error del intento anterior.

In [ ]:
import json, time
from paper_replica.train import entrenar

A_ENTRENAR = ('CB55',)          # ('CB55',) para la corrida corta; MANUSCRITOS para las tres

resultados = {}
for m in A_ENTRENAR:
    ckpt = SALIDA / f'{m}.pt'
    if ckpt.exists():
        # no reentrenar lo que ya esta: permite retomar si el runtime se corta
        resultados[m] = {'state_dict': torch.load(ckpt, map_location='cpu')}
        print(f'{m}: reusando {ckpt.name}')
        continue

    t0 = time.time()
    r = entrenar(m, cfg, device='cuda', log_cada=5)
    torch.save(r['state_dict'], ckpt)
    (SALIDA / f'{m}_historial.json').write_text(json.dumps(r['historial'], indent=2))
    resultados[m] = r
    print(f'{m}: mejor epoca {r["mejor_epoca"]}  val {r["mejor_val_loss"]:.4f}  '
          f'{(time.time() - t0) / 60:.1f} min  -> {ckpt}\n')
    torch.cuda.empty_cache()

## 6. Evaluacion y veredicto

El veredicto usa **la metrica del repo oficial**: `get_scores(average="weighted")` de
`function.py`, es decir sklearn sobre todos los pixeles del split juntos, con las 4 clases
y el fondo incluido. Es lo que produce la tabla del README.

Las variantes A y B del F1 del paper (diseno §8) quedan abajo como diagnostico. **No son
comparables con el README**: promedian macro sobre las 3 clases de primer plano y excluyen
el fondo, que es ~82 % de los pixeles. Confundirlas costo la corrida del 2026-08-11.

Se comparan las cuatro cifras publicadas, no solo el F1: reproducir el F1 y errarle al IoU
significaria que la agregacion todavia no es la correcta.

La matriz de confusion se guarda en Drive. Con ella cualquier metrica se recalcula despues
sin GPU.

In [ ]:
from paper_replica.report import evaluar_manuscrito, imprimir_veredicto

evaluaciones = []
for m in resultados:
    e = evaluar_manuscrito(m, resultados[m]['state_dict'], cfg, device='cuda')
    # lo primero que se persiste, antes de imprimir nada: es lo que permite
    # recalcular cualquier metrica despues sin volver a tocar la GPU
    (SALIDA / f'{m}_evaluacion.json').write_text(json.dumps(e, indent=2))
    evaluaciones.append(e)

imprimir_veredicto(evaluaciones)

In [ ]:
for e in evaluaciones:
    o = e['oficial']
    print(f"=== {e['manuscrito']} ===")
    print('  ponderado :', {k: round(v, 4) for k, v in o['weighted'].items()})
    print('  macro     :', {k: round(v, 4) for k, v in o['macro'].items()})
    print('  accuracy  :', round(o['accuracy'], 4))
    print('  F1 por clase (0=fondo, 1=comment, 2=decoration, 3=text):')
    for c, (f1, sop) in enumerate(zip(o['por_clase']['f1'], o['soporte'])):
        print(f'    {c}  f1={f1:.4f}  {sop:>10d} px  ({100 * sop / sum(o["soporte"]):.2f} %)')
    print('  --- diagnostico, NO comparable con el README ---')
    print('  variante A:', {k: round(v, 4) for k, v in e['f1_variante_a'].items()})
    print('  variante B:', {k: round(v, 4) for k, v in e['f1_variante_b'].items()})
    print('  max logit :', f"{e['max_logit']:.1f}")
    print('  confusion (filas=gt, columnas=pred):')
    for fila in e['confusion']:
        print('   ', '  '.join(f'{v:>10d}' for v in fila))
    print()

## 7. Ablacion del refinamiento

Responde dos preguntas abiertas del diseno: cuanto aporta Sauvola, y si el `k` correcto es
el del codigo oficial (0.01) o el del paper 2024 (0.2). Se mide con la metrica oficial.

Se incluye `removeSmallCC` como tercera variante: `test.py` oficial lo importa pero **nunca
lo llama**, asi que el default del paquete es `min_size=0`. La fila esta para cuantificar
cuanto cambiaria si se activara, no porque replique nada.

In [ ]:
from dataclasses import replace
from paper_replica.report import METRICAS_OFICIALES

print(f"{'manuscrito':<10} {'variante':<22} {'f1 pond.':>9} {'iou pond.':>10} "
      f"{'f1 oficial':>11} {'delta':>8}")
print('-' * 74)
for m in resultados:
    sd = resultados[m]['state_dict']
    variantes = [
        ('sin refinar',            replace(cfg),                 False, 0),
        ('sauvola k=0.01',         replace(cfg, sauvola_k=0.01), True,  0),
        ('sauvola k=0.2',          replace(cfg, sauvola_k=0.2),  True,  0),
        ('sauvola k=0.01 + smallCC', replace(cfg, sauvola_k=0.01), True, 64),
    ]
    for nombre, c, refinar_flag, min_size in variantes:
        r = evaluar_manuscrito(m, sd, c, device='cuda',
                               refinamiento=refinar_flag, min_size=min_size)
        w = r['oficial']['weighted']
        pub = METRICAS_OFICIALES[m]['f1']
        print(f"{m:<10} {nombre:<22} {w['f1']:>9.4f} {w['iou']:>10.4f} "
              f"{pub:>11.3f} {w['f1'] - pub:>+8.4f}")

## 8. Que hacer con el resultado

**Si pasa:** correr los tres manuscritos (`A_ENTRENAR = MANUSCRITOS`), anotar los numeros en
`docs/replica-denardin/resultados-fase-1.md` y avanzar a la Fase 2.

**Si no pasa:** no avanzar. Sospechosos en orden, todos localizables porque los datos y los
hiperparametros son los oficiales:

1. El `softmax` espurio del codigo oficial antes de la CE — probar replicando el bug
2. `k` de Sauvola y el refinamiento (celda 7)
3. La interpolacion del resize (bilineal imagen / vecino mas cercano GT)
4. `micro_batch` afectando a BatchNorm — comparar 8 contra 4

Ya descartados leyendo el codigo de los autores, no adivinando:

- **la agregacion de la metrica** — era el problema del 2026-08-11; ahora se usa
  `average="weighted"` sobre las 4 clases, igual que `get_scores`
- **`removeSmallCC`** — `test.py` lo importa y nunca lo llama; el default es `min_size=0`

Pase o no pase, los checkpoints y las matrices de confusion quedan en Drive.